# The overfitting problem

![](<src/09_Table_The Overfitting.png>)

## Load the data

In [9]:
import pandas as pd

df = pd.read_excel('data/Microsoft_LinkedIn_Processed.xlsx', parse_dates=['Date'], index_col=0)
df

,Close,High,Low,Open,Volume,change_tomorrow,change_tomorrow_direction
Date,,,,,,,
2016-12-08,61.009998,61.580002,60.840000,61.299999,21220800,1.549141,UP
2016-12-09,61.970001,61.990002,61.130001,61.180000,27349400,0.321694,UP
2016-12-12,62.169998,62.299999,61.720001,61.820000,20198100,1.286125,UP
2016-12-13,62.980000,63.419998,62.240002,62.500000,35718900,-0.478620,DOWN
2016-12-14,62.680000,63.450001,62.529999,63.000000,30352700,-0.159793,DOWN
...,...,...,...,...,...,...,...
2026-05-14,409.429993,411.839996,400.880005,404.480011,27077500,2.960282,UP
2026-05-15,421.920013,428.170013,412.910004,414.269989,50771200,0.382489,UP
2026-05-18,423.540009,425.119995,415.609985,416.619995,32564100,-1.466148,DOWN


## Machine Learning Model

### Separate the data

1. Target: which variable do you want to predict?
2. Explanatory: which variables will you use to calculate the prediction?

In [10]:
target = df.change_tomorrow
explanatory = df[['Open','High','Low','Close','Volume']]

## Train test split

### Split the dataset

In [11]:
n_days = len(df.index)
n_days

2374

In [12]:
n_days_split = int(n_days*0.7)
n_days_split

1661

In [13]:
X_train, y_train = explanatory.iloc[:n_days_split], target.iloc[:n_days_split]
X_test, y_test = explanatory.iloc[n_days_split:], target.iloc[n_days_split:]

In [14]:
X_train

,Open,High,Low,Close,Volume
Date,,,,,
2016-12-08,61.299999,61.580002,60.840000,61.009998,21220800
2016-12-09,61.180000,61.990002,61.130001,61.970001,27349400
2016-12-12,61.820000,62.299999,61.720001,62.169998,20198100
2016-12-13,62.500000,63.419998,62.240002,62.980000,35718900
2016-12-14,63.000000,63.450001,62.529999,62.680000,30352700
...,...,...,...,...,...
2023-07-12,336.600006,341.649994,335.670013,337.200012,29995300
2023-07-13,339.559998,343.739990,339.019989,342.660004,20567200
2023-07-14,347.589996,351.429993,344.309998,345.239990,28352700


In [15]:
X_test

,Open,High,Low,Close,Volume
Date,,,,,
2023-07-19,361.750000,362.459991,352.440002,355.079987,39732900
2023-07-20,353.570007,357.970001,345.369995,346.869995,33778400
2023-07-21,349.149994,350.299988,339.829987,343.769989,69405400
2023-07-24,345.850006,346.920013,342.309998,345.109985,26678100
2023-07-25,347.109985,351.890015,345.070007,350.980011,41637700
...,...,...,...,...,...
2026-05-14,404.480011,411.839996,400.880005,409.429993,27077500
2026-05-15,414.269989,428.170013,412.910004,421.920013,50771200
2026-05-18,416.619995,425.119995,415.609985,423.540009,32564100


### Fit the model on train set # We are training mathematical equation on the training set

In [16]:
from sklearn.tree import DecisionTreeRegressor

In [17]:
model_dt_split = DecisionTreeRegressor(max_depth=15, random_state=42)

In [18]:
model_dt_split.fit(X=X_train, y=y_train)

DecisionTreeRegressor(max_depth=15, random_state=42)

### Evaluate model

#### On test set

In [19]:
from sklearn.metrics import mean_squared_error

y_pred_test = model_dt_split.predict(X=X_test)
mean_squared_error(y_true=y_test, y_pred=y_pred_test)

8.72777972927931

#### On train set

In [20]:
y_pred_train = model_dt_split.predict(X=X_train)
mean_squared_error(y_true=y_train, y_pred=y_pred_train)

2.477095984197048

## [ ] Backtesting

In [27]:
from backtesting import Backtest, Strategy

### Create the `Strategy`

In [22]:
class Regression(Strategy):
    limit_buy = 1
    limit_sell = -5
    
    def init(self):
        self.model = DecisionTreeRegressor(max_depth=15, random_state=42)
        self.already_bought = False
        
        self.model.fit(X=X_train, y=y_train)

    def next(self):
        explanatory_today = self.data.df.iloc[[-1], :]
        forecast_tomorrow = self.model.predict(explanatory_today)[0]
        
        if forecast_tomorrow > self.limit_buy and self.already_bought == False:
            self.buy()
            self.already_bought = True
        elif forecast_tomorrow < self.limit_sell and self.already_bought == True:
            self.sell()
            self.already_bought = False
        else:
            pass

### Run the backtest on `test` data

In [23]:
bt = Backtest(X_test, Regression,
              cash=10000, commission=.002, exclusive_orders=True)

In [24]:
results = bt.run(limit_buy=1, limit_sell=-5)

df_results_test = results.to_frame(name='Values').loc[:'Return [%]']\
    .rename({'Values':'Out of Sample (Test)'}, axis=1)
df_results_test

,Out of Sample (Test)
Start,2023-07-19 00:00:00
End,2026-05-20 00:00:00
Duration,1036 days 00:00:00
Exposure Time [%],99.018233
Equity Final [$],12315.147139
Equity Peak [$],16024.246962
Return [%],23.151471


### Run the backtest on `train` data

In [25]:
bt = Backtest(X_train, Regression,
              cash=10000, commission=.002, exclusive_orders=True)

results = bt.run(limit_buy=1, limit_sell=-5)

df_results_train = results.to_frame(name='Values').loc[:'Return [%]']\
    .rename({'Values':'In Sample (Train)'}, axis=1)
df_results_train

,In Sample (Train)
Start,2016-12-08 00:00:00
End,2023-07-18 00:00:00
Duration,2413 days 00:00:00
Exposure Time [%],51.294401
Equity Final [$],21848.144185
Equity Peak [$],23339.745465
Return [%],118.481442


### Compare both backtests

In [26]:
df_results = pd.concat([df_results_test, df_results_train], axis=1)
df_results

,Out of Sample (Test),In Sample (Train)
Start,2023-07-19 00:00:00,2016-12-08 00:00:00
End,2026-05-20 00:00:00,2023-07-18 00:00:00
Duration,1036 days 00:00:00,2413 days 00:00:00
Exposure Time [%],99.018233,51.294401
Equity Final [$],12315.147139,21848.144185
Equity Peak [$],16024.246962,23339.745465
Return [%],23.151471,118.481442


## Practice to master the knowledge

Work on the challenge with another dataset:

1. Learn the <a>mental models</a> to solve the challenge faster.
2. Complete the <a href="09D_The Overfitting Problem.ipynb">notebook</a>.